# DyT-ViT: Dynamic Tanh Vision Transformer on CIFAR-100

Three models, one recipe: the same split, seed (42), backbone, optimizer, and epoch budget. Only the normalization layer changes.

| Model | Normalization | $\alpha_0$ | Role |
|---|---|---|---|
| A — `LayerNorm-ViT` | `nn.LayerNorm` | — | Baseline |
| B — `DyT-ViT` | `DynamicTanh` | 0.5 | Replace every LayerNorm with DyT |
| C — `DyT-ViT-Var` | `DynamicTanh` | 1.0 | Same model as B, with a steeper starting $\alpha$ |


## Setup
Imports and the device. Mixed precision is used only when a GPU is available. Full training is meant to run on a Colab GPU; a CPU is enough for the checks in this notebook.


In [ ]:
import os, sys, json, time, random, pickle, shutil, warnings, gc, zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support

# GradScaler skips the optimizer step on a bad batch, and the per-step scheduler then warns. That warning is harmless.
warnings.filterwarnings("ignore", message=r"Detected call of `lr_scheduler.step\(\)`")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | torchvision {torchvision.__version__}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | {props.total_memory / 1024**3:.1f} GB | AMP enabled")
else:
    print("No CUDA device found. Training the full run needs a Colab GPU.")


### Why Google Drive
A Colab machine is erased when the runtime disconnects, so every checkpoint, history, plot, and table is written to `MyDrive/DyT_ViT_CIFAR100_Runs`. The dataset is not downloaded: upload `cifar-100-python.zip` to Drive once. The public CIFAR host is too slow to rely on from Colab. Locally, outputs go to `./runs` and the data is read from `./data`.


In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_SAVE_DIR = "/content/drive/MyDrive/DyT_ViT_CIFAR100_Runs"
    DRIVE_CIFAR_ZIP = "/content/drive/MyDrive/cifar-100-python.zip"
else:
    BASE_SAVE_DIR = "./runs"
    DRIVE_CIFAR_ZIP = None

os.makedirs(BASE_SAVE_DIR, exist_ok=True)
print(f"Running in Colab: {IN_COLAB} | BASE_SAVE_DIR = {os.path.abspath(BASE_SAVE_DIR)}")
if DRIVE_CIFAR_ZIP:
    print(f"DRIVE_CIFAR_ZIP = {DRIVE_CIFAR_ZIP} | exists={os.path.isfile(DRIVE_CIFAR_ZIP)}")


## Configuration
One dictionary holds the hyperparameters shared by all three models. `DEBUG_SMOKE_TEST = True` trains for 2 epochs on a small subset and checks that a run can resume. Leave it `False` for the real experiment.


In [ ]:
DEBUG_SMOKE_TEST = False

CONFIG = {
    "seed": 42,
    "data_root": "./data",
    "num_classes": 100,
    "val_size": 5000,
    "mean": (0.5071, 0.4867, 0.4408),
    "std": (0.2675, 0.2565, 0.2761),
    "batch_size": 128,
    "num_workers": 2,
    "pin_memory": True,
    "img_size": 32,
    "patch_size": 4,
    "in_chans": 3,
    "embed_dim": 192,
    "depth": 6,
    "num_heads": 6,
    "mlp_ratio": 4.0,
    "dropout": 0.1,
    "alpha_init_standard": 0.5,
    "alpha_init_variation": 1.0,
    "epochs": 100,
    "warmup_epochs": 5,
    "lr": 1e-3,
    "min_lr": 1e-5,
    "warmup_start_factor": 0.01,
    "weight_decay": 0.05,
    "label_smoothing": 0.1,
    "smoke_subset": {"train": 1024, "val": 512, "test": 512},
}

if DEBUG_SMOKE_TEST:
    CONFIG.update(epochs=2, warmup_epochs=1)

SAVE_DIR = os.path.join(BASE_SAVE_DIR, "smoke_test") if DEBUG_SMOKE_TEST else BASE_SAVE_DIR
if DEBUG_SMOKE_TEST and os.path.isdir(SAVE_DIR):
    shutil.rmtree(SAVE_DIR)
PLOT_DIR = os.path.join(SAVE_DIR, "plots")

EXPERIMENTS = {
    "ln_vit":      {"display_name": "LayerNorm-ViT",        "norm_type": "layernorm", "alpha_init": None},
    "dyt_vit_a05": {"display_name": "DyT-ViT (α₀=0.5)",     "norm_type": "dyt",       "alpha_init": CONFIG["alpha_init_standard"]},
    "dyt_vit_a10": {"display_name": "DyT-ViT-Var (α₀=1.0)", "norm_type": "dyt",       "alpha_init": CONFIG["alpha_init_variation"]},
}
for key in EXPERIMENTS:
    os.makedirs(os.path.join(SAVE_DIR, key), exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)


def set_seed(seed):
    """Set the Python, NumPy, and PyTorch seeds so the three models start from the same place."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CONFIG["seed"])
print(f"SMOKE TEST: {DEBUG_SMOKE_TEST} | epochs={CONFIG['epochs']} | SAVE_DIR = {os.path.abspath(SAVE_DIR)}")
print("Model sub-directories:", sorted(os.listdir(SAVE_DIR)))


### CIFAR-100 from the Drive zip
The notebook never downloads CIFAR-100. It copies the zip from Drive into `./data` and extracts `cifar-100-python/{train,test,meta}`. If that folder is already on the machine, this step does nothing.


In [ ]:
def cifar100_ready(root):
    """Return True when the train, test, and meta files are already extracted."""
    base = os.path.join(root, "cifar-100-python")
    return all(os.path.isfile(os.path.join(base, name)) for name in ("train", "test", "meta"))


def _normalize_cifar_layout(root):
    """Move an extracted archive into data/cifar-100-python, which is the folder torchvision expects."""
    base = os.path.join(root, "cifar-100-python")
    if cifar100_ready(root):
        return
    if all(os.path.isfile(os.path.join(root, name)) for name in ("train", "test", "meta")):
        os.makedirs(base, exist_ok=True)
        for name in ("train", "test", "meta", "file.txt"):
            src = os.path.join(root, name)
            if os.path.isfile(src):
                shutil.move(src, os.path.join(base, name))
        return
    for entry in os.listdir(root):
        cand = os.path.join(root, entry)
        if os.path.isdir(cand) and all(os.path.isfile(os.path.join(cand, name)) for name in ("train", "test", "meta")):
            if os.path.abspath(cand) != os.path.abspath(base):
                if os.path.exists(base):
                    shutil.rmtree(base)
                shutil.move(cand, base)
            return


def _extract_zip(archive_path, dest_dir):
    """Unzip an archive into dest_dir."""
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(dest_dir)


def prepare_cifar100(data_root, drive_zip=None):
    """Extract the CIFAR-100 zip if the files are not already present. On Colab the zip is copied from Drive first."""
    os.makedirs(data_root, exist_ok=True)
    if cifar100_ready(data_root):
        print(f"CIFAR-100 already present at {os.path.join(data_root, 'cifar-100-python')}")
        return

    candidates = []
    if drive_zip:
        candidates.append(drive_zip)
    candidates.extend([
        os.path.join(data_root, "cifar-100-python.zip"),
        "./cifar-100-python.zip",
    ])
    archive = next((p for p in candidates if p and os.path.isfile(p)), None)
    if archive is None:
        hint = drive_zip or "./cifar-100-python.zip"
        raise FileNotFoundError(
            f"CIFAR-100 not found under {data_root} and no zip at {hint}. "
            "Upload cifar-100-python.zip to Google Drive (or place it next to the notebook), then re-run."
        )

    local_archive = os.path.join(data_root, os.path.basename(archive))
    if os.path.abspath(archive) != os.path.abspath(local_archive):
        print(f"Copying archive: {archive} -> {local_archive}")
        shutil.copy2(archive, local_archive)
    else:
        print(f"Using local archive: {local_archive}")

    print(f"Extracting {local_archive} ...")
    _extract_zip(local_archive, data_root)
    _normalize_cifar_layout(data_root)
    assert cifar100_ready(data_root), (
        f"Extracted archive but missing train/test/meta under {os.path.join(data_root, 'cifar-100-python')}"
    )
    print(f"Ready: {os.path.join(data_root, 'cifar-100-python')}")


prepare_cifar100(CONFIG["data_root"], drive_zip=DRIVE_CIFAR_ZIP)


## Data

### Raw files
Each training image is stored as 3,072 bytes: 1,024 red, then 1,024 green, then 1,024 blue. Every image has a fine label (100 classes) and a coarse label (20 superclasses). The fine-to-coarse map is kept for the confusion matrix later.


In [ ]:
def inspect_raw_cifar100(root):
    """Read the CIFAR pickle, print its layout, and return class names plus the fine-to-coarse map.
    The files are opened locally. Nothing is downloaded.
    """
    assert cifar100_ready(root), f"Run prepare_cifar100 first; missing files under {root}/cifar-100-python"
    datasets.CIFAR100(root=root, train=True, download=False)
    base = os.path.join(root, "cifar-100-python")
    with open(os.path.join(base, "train"), "rb") as f:
        raw = pickle.load(f, encoding="bytes")
    with open(os.path.join(base, "meta"), "rb") as f:
        meta = pickle.load(f, encoding="bytes")

    data = raw[b"data"]
    fine = np.asarray(raw[b"fine_labels"])
    coarse = np.asarray(raw[b"coarse_labels"])

    print("Keys:", list(raw.keys()))
    print(f"b'data'          shape={data.shape} dtype={data.dtype} range=[{data.min()}, {data.max()}]")
    print(f"b'fine_labels'   n={len(fine)} classes={len(np.unique(fine))} images/class={np.bincount(fine).min()}-{np.bincount(fine).max()}")
    print(f"b'coarse_labels' n={len(coarse)} classes={len(np.unique(coarse))} images/class={np.bincount(coarse).min()}-{np.bincount(coarse).max()}")

    img0 = data[0].reshape(3, 32, 32)
    print(f"First image reshaped to {img0.shape}; per-channel means = {img0.reshape(3, -1).mean(1).round(1)}")

    fine_names = [n.decode() for n in meta[b"fine_label_names"]]
    coarse_names = [n.decode() for n in meta[b"coarse_label_names"]]

    fine_to_coarse = np.zeros(100, dtype=np.int64)
    fine_to_coarse[fine] = coarse
    assert all(len(np.unique(coarse[fine == c])) == 1 for c in range(100)), "each fine class must map to one superclass"
    return fine_names, coarse_names, fine_to_coarse


FINE_NAMES, COARSE_NAMES, FINE_TO_COARSE = inspect_raw_cifar100(CONFIG["data_root"])
print(f"Example mapping: '{FINE_NAMES[0]}' -> '{COARSE_NAMES[FINE_TO_COARSE[0]]}'")


### Train, validation, and test
The 50,000 training images are split once, with seed 42, into 45,000 train and 5,000 validation. The training set is loaded twice: one copy is randomly cropped and flipped, and the other is only normalized. Validation uses the second copy, so augmentation cannot leak into the score used to pick the best epoch. The 10,000 test images are used only at the end.


In [ ]:
def get_cifar100_dataloaders(config, smoke=False):
    """Build the train, validation, and test loaders from the fixed split.
    Train batches are augmented. Validation and test batches are only normalized.
    """
    norm = transforms.Normalize(mean=config["mean"], std=config["std"])
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        norm,
    ])
    eval_tf = transforms.Compose([transforms.ToTensor(), norm])

    root = config["data_root"]
    full_train_aug = datasets.CIFAR100(root, train=True, download=False, transform=train_tf)
    full_train_eval = datasets.CIFAR100(root, train=True, download=False, transform=eval_tf)
    test_set = datasets.CIFAR100(root, train=False, download=False, transform=eval_tf)

    g = torch.Generator().manual_seed(config["seed"])
    perm = torch.randperm(len(full_train_aug), generator=g).tolist()
    val_idx, train_idx = perm[:config["val_size"]], perm[config["val_size"]:]
    test_idx = list(range(len(test_set)))
    assert not set(train_idx) & set(val_idx), "train/val overlap"

    if smoke:
        s = config["smoke_subset"]
        train_idx, val_idx, test_idx = train_idx[:s["train"]], val_idx[:s["val"]], test_idx[:s["test"]]

    splits = {
        "train": Subset(full_train_aug, train_idx),
        "val": Subset(full_train_eval, val_idx),
        "test": Subset(test_set, test_idx),
    }
    use_pin = config["pin_memory"] and torch.cuda.is_available()
    workers = config["num_workers"]
    loader_kw = dict(batch_size=config["batch_size"], num_workers=workers, pin_memory=use_pin,
                     persistent_workers=workers > 0)
    loaders = {
        "train": DataLoader(splits["train"], shuffle=True, drop_last=True, **loader_kw),
        "val": DataLoader(splits["val"], shuffle=False, **loader_kw),
        "test": DataLoader(splits["test"], shuffle=False, **loader_kw),
    }
    info = {"full_train_aug": full_train_aug, "full_train_eval": full_train_eval,
            "train_idx": train_idx, "val_idx": val_idx, "test_idx": test_idx}
    return loaders, info


loaders, data_info = get_cifar100_dataloaders(CONFIG, smoke=DEBUG_SMOKE_TEST)

with open(os.path.join(SAVE_DIR, "split_indices.json"), "w") as f:
    json.dump({"seed": CONFIG["seed"], "train_idx": data_info["train_idx"], "val_idx": data_info["val_idx"]}, f)

val_labels = np.asarray(data_info["full_train_eval"].targets)[data_info["val_idx"]]
print(f"Train: {len(data_info['train_idx']):,} | Val: {len(data_info['val_idx']):,} | Test: {len(data_info['test_idx']):,}")
print(f"Train/val overlap: {len(set(data_info['train_idx']) & set(data_info['val_idx']))} images")
print(f"Val images per class: min={np.bincount(val_labels, minlength=100).min()} max={np.bincount(val_labels, minlength=100).max()}")

xb, yb = next(iter(loaders["train"]))
print(f"Train batch: images {list(xb.shape)} {xb.dtype} | labels {list(yb.shape)} {yb.dtype} | steps/epoch = {len(loaders['train'])}")


### Sample images
The top row is the image the validator sees. The two rows under it are two random augmentations of the same training image.


In [ ]:
MEAN_T = torch.tensor(CONFIG["mean"]).view(3, 1, 1)
STD_T = torch.tensor(CONFIG["std"]).view(3, 1, 1)


def denormalize(img):
    """Undo channel normalization so a tensor can be drawn with ordinary colors."""
    return (img * STD_T + MEAN_T).clamp(0, 1)


n_show = 8
sample_ids = data_info["train_idx"][:n_show]
fig, axes = plt.subplots(3, n_show, figsize=(2 * n_show, 6.3))
row_titles = ["clean (eval tf)", "augmented draw 1", "augmented draw 2"]
for col, idx in enumerate(sample_ids):
    clean, label = data_info["full_train_eval"][idx]
    views = [clean, data_info["full_train_aug"][idx][0], data_info["full_train_aug"][idx][0]]
    for row, img in enumerate(views):
        ax = axes[row, col]
        ax.imshow(denormalize(img).permute(1, 2, 0).numpy())
        ax.set_xticks([]); ax.set_yticks([])
        if row == 0:
            ax.set_title(f"{FINE_NAMES[label]}\n({COARSE_NAMES[FINE_TO_COARSE[label]]})", fontsize=8)
        if col == 0:
            ax.set_ylabel(row_titles[row], fontsize=9)
plt.suptitle("CIFAR-100 training samples: clean vs. random augmentation", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "cifar100_samples.png"), dpi=150, bbox_inches="tight")
plt.show()


## Dynamic Tanh

DyT replaces LayerNorm. It does not compute a mean or a variance. It squashes each value on its own:

$$\mathrm{DyT}(x) = \gamma \odot \tanh(\alpha x) + \beta$$

$\alpha$ is one learnable number per layer. It starts at 0.5 in model B and at 1.0 in model C. $\gamma$ and $\beta$ are per channel, as in LayerNorm, so a DyT layer has one more parameter than LayerNorm.


In [ ]:
class DynamicTanh(nn.Module):
    def __init__(self, normalized_shape, alpha_init_value=0.5):
        """Create one layer. Alpha starts at alpha_init_value. The scale starts at 1 and the shift at 0."""
        super().__init__()
        self.normalized_shape = normalized_shape
        self.alpha_init_value = alpha_init_value
        self.alpha = nn.Parameter(torch.ones(1) * alpha_init_value)
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x):
        """Squash each value with tanh(alpha * x), then apply the per-channel scale and shift."""
        x = torch.tanh(self.alpha * x)
        return x * self.weight + self.bias

    def extra_repr(self):
        """Show the width and the starting alpha when the module is printed."""
        return f"normalized_shape={self.normalized_shape}, alpha_init_value={self.alpha_init_value}"


print(DynamicTanh(192))


### Check against the formula
On one batch we check that DyT matches $\gamma \tanh(\alpha x)+\beta$, that it has $2C+1$ parameters against $2C$ for LayerNorm, and that gradients reach $\alpha$, $\gamma$, $\beta$, and the input.


In [ ]:
torch.manual_seed(0)
B, N, C = 4, 65, 192
x = (torch.randn(B, N, C) * 4.0 + 1.0).requires_grad_(True)

dyt = DynamicTanh(C, alpha_init_value=0.5)
ln = nn.LayerNorm(C)
y_dyt, y_ln = dyt(x), ln(x)

assert y_dyt.shape == y_ln.shape == x.shape, "shape mismatch"
n_dyt = sum(p.numel() for p in dyt.parameters())
n_ln = sum(p.numel() for p in ln.parameters())
assert n_dyt == 2 * C + 1 and n_ln == 2 * C, (n_dyt, n_ln)
assert torch.allclose(y_dyt, dyt.weight * torch.tanh(dyt.alpha * x) + dyt.bias)
assert y_dyt.abs().max() <= 1.0, "DyT output must lie in [-1, 1] at init"

print(f"Shapes: input {list(x.shape)} -> DyT {list(y_dyt.shape)}, LN {list(y_ln.shape)}")
print(f"Params: DyT = {n_dyt} (2C+1) | LayerNorm = {n_ln} (2C)")
print(f"Input     range [{x.min():7.3f}, {x.max():7.3f}]  mean {x.mean():6.3f}  std {x.std():6.3f}")
print(f"DyT out   range [{y_dyt.min():7.3f}, {y_dyt.max():7.3f}]  mean {y_dyt.mean():6.3f}  std {y_dyt.std():6.3f}  (bounded by tanh)")
print(f"LN out    range [{y_ln.min():7.3f}, {y_ln.max():7.3f}]  per-token mean {y_ln.mean(-1).abs().max():.1e}  per-token std {y_ln.std(-1, unbiased=False).mean():.3f}")

loss = (y_dyt * torch.randn_like(y_dyt)).sum()
loss.backward()
for name, grad in [("alpha", dyt.alpha.grad), ("weight", dyt.weight.grad), ("bias", dyt.bias.grad), ("x", x.grad)]:
    assert grad is not None and grad.abs().sum() > 0, f"zero/missing gradient on {name}"
    print(f"grad[{name:6s}] shape {list(grad.shape)}  |grad| sum = {grad.abs().sum():.4f}")
print("DynamicTanh unit test PASSED")

### Why the two curves look alike
A single token through LayerNorm is a straight line, and tokens with different spreads have different slopes. Drawn together, those lines form an S. The right plot is the S that DyT uses at the start of training.


In [ ]:
with torch.no_grad():
    scales = torch.tensor([0.5, 1.0, 2.0, 4.0]).view(-1, 1)
    tokens = torch.randn(4, C) * scales
    ln_out = nn.LayerNorm(C, elementwise_affine=False)(tokens)
    xs = torch.linspace(-8, 8, 400)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i in range(4):
    axes[0].scatter(tokens[i], ln_out[i], s=4, label=f"token std ≈ {scales[i].item():g}")
axes[0].set(title="LayerNorm: per-token linear maps", xlabel="input", ylabel="output")
axes[0].legend(fontsize=8)
for a0 in (CONFIG["alpha_init_standard"], CONFIG["alpha_init_variation"]):
    axes[1].plot(xs, torch.tanh(a0 * xs), label=f"tanh({a0}·x)")
axes[1].set(title="DyT at initialization (γ=1, β=0)", xlabel="input", ylabel="output")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "dyt_vs_layernorm_mapping.png"), dpi=150, bbox_inches="tight")
plt.show()


## Vision Transformer

The block is pre-norm: the norm sits inside the branch, and the skip connection is added afterwards.

$$x \leftarrow x + \mathrm{MHSA}(\mathrm{Norm}(x)), \qquad x \leftarrow x + \mathrm{MLP}(\mathrm{Norm}(x))$$

Models A, B, and C are the same class. A small factory inserts either LayerNorm or DyT, so the only change is those 13 layers: 2 in each of the 6 blocks, plus 1 before the classifier. A 32×32 image becomes 64 patches, then 65 tokens after the class token is added.


In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_chans, embed_dim):
        """Prepare a convolution that both cuts the image into patches and projects each patch."""
        super().__init__()
        assert img_size % patch_size == 0
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        """Turn one image into a sequence of patch vectors."""
        x = self.proj(x)
        return x.flatten(2).transpose(1, 2)


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, num_heads, attn_drop=0.0, proj_drop=0.0):
        """Set up the query, key, and value projection for the given number of heads."""
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        """Let every token attend to the others, then project the result back to the model width."""
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = self.attn_drop(attn.softmax(dim=-1))
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.proj(x))


class MLP(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.0):
        """Two linear layers with GELU between them. The hidden layer is wider than the model."""
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        """Expand the token, apply GELU, and project it back."""
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio, dropout, norm_factory):
        """One pre-norm block. norm_factory builds the LayerNorm or DyT used in this model."""
        super().__init__()
        self.norm1 = norm_factory(dim)
        self.attn = MultiHeadSelfAttention(dim, num_heads, attn_drop=dropout, proj_drop=dropout)
        self.norm2 = norm_factory(dim)
        self.mlp = MLP(dim, int(dim * mlp_ratio), dropout)

    def forward(self, x):
        """Add the attention output, then add the MLP output. The skip path itself is not normalized."""
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class VisionTransformer(nn.Module):
    def __init__(self, img_size, patch_size, in_chans, num_classes, embed_dim, depth,
                 num_heads, mlp_ratio, dropout, norm_factory):
        """Stack the patch embedding, class token, blocks, final norm, and the classifier."""
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout, norm_factory) for _ in range(depth)
        ])
        self.norm = norm_factory(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        """Initialize linear layers with a small truncated normal. LayerNorm keeps its own default."""
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x):
        """Embed the image, run the blocks, and classify from the class token."""
        x = self.patch_embed(x)
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return self.head(x[:, 0])


def make_norm_factory(norm_type, alpha_init=None):
    """Return a function that builds either LayerNorm or Dynamic Tanh for a given width."""
    if norm_type == "layernorm":
        return lambda dim: nn.LayerNorm(dim, eps=1e-6)
    if norm_type == "dyt":
        return lambda dim: DynamicTanh(dim, alpha_init_value=alpha_init)
    raise ValueError(f"unknown norm_type: {norm_type}")


def build_model(norm_type, alpha_init, config):
    """Build one Vision Transformer from the shared config and the chosen norm."""
    return VisionTransformer(
        img_size=config["img_size"], patch_size=config["patch_size"], in_chans=config["in_chans"],
        num_classes=config["num_classes"], embed_dim=config["embed_dim"], depth=config["depth"],
        num_heads=config["num_heads"], mlp_ratio=config["mlp_ratio"], dropout=config["dropout"],
        norm_factory=make_norm_factory(norm_type, alpha_init),
    )


def count_parameters(model):
    """Count the trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


### Same weights, different norm
All three models are built from the same seed. Shared tensors should match exactly. A DyT model should contain no LayerNorm, and it should have 13 extra parameters, one alpha per norm layer.


In [ ]:
dummy = torch.randn(4, 3, 32, 32)
test_models, param_counts = {}, {}
for key, spec in EXPERIMENTS.items():
    set_seed(CONFIG["seed"])
    model = build_model(spec["norm_type"], spec["alpha_init"], CONFIG).eval()
    with torch.no_grad():
        out = model(dummy)
    assert out.shape == (4, CONFIG["num_classes"]), out.shape

    n_ln = sum(isinstance(m, nn.LayerNorm) for m in model.modules())
    n_dyt = sum(isinstance(m, DynamicTanh) for m in model.modules())
    expected_norms = 2 * CONFIG["depth"] + 1
    if spec["norm_type"] == "dyt":
        assert n_ln == 0 and n_dyt == expected_norms, (n_ln, n_dyt)
        alphas = {m.alpha.item() for m in model.modules() if isinstance(m, DynamicTanh)}
        assert alphas == {spec["alpha_init"]}, alphas
    else:
        assert n_ln == expected_norms and n_dyt == 0, (n_ln, n_dyt)

    test_models[key] = model
    param_counts[key] = count_parameters(model)
    print(f"{spec['display_name']:24s} out {list(out.shape)} | LayerNorm={n_ln:2d} DyT={n_dyt:2d} | params = {param_counts[key]:,}")

assert param_counts["dyt_vit_a05"] - param_counts["ln_vit"] == 2 * CONFIG["depth"] + 1
assert param_counts["dyt_vit_a05"] == param_counts["dyt_vit_a10"]

ref_sd = test_models["ln_vit"].state_dict()
for key in ("dyt_vit_a05", "dyt_vit_a10"):
    sd = test_models[key].state_dict()
    shared = [k for k in ref_sd if k in sd]
    assert all(torch.equal(ref_sd[k], sd[k]) for k in shared), f"init mismatch in {key}"
    print(f"{key}: {len(shared)} shared tensors identical to LayerNorm-ViT; extra = {sorted(set(sd) - set(ref_sd))[:3]} ...")

print("\nLayerNorm-ViT module tree:")
print(test_models["ln_vit"].blocks[0])
del test_models
print("ViT architecture unit test PASSED")

## Training

AdamW uses weight decay 0.05 on weight matrices only. Biases and normalization parameters, including $\alpha$, are left alone. Decaying $\alpha$ would pull it toward zero and shrink the tanh, which is not the regularizer we want.

The learning rate rises for 5 epochs and then follows a cosine down to `min_lr`, updated after every batch. The loss is cross-entropy with label smoothing 0.1. On a GPU the forward and backward pass use float16.


In [ ]:
def build_optimizer(model, config):
    """AdamW with decay on matrices only. One-dimensional parameters, including alpha, are not decayed."""
    decay, no_decay = [], []
    for _, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (no_decay if p.ndim <= 1 else decay).append(p)
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": config["weight_decay"]},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=config["lr"], betas=(0.9, 0.999),
    )


def build_scheduler(optimizer, config, steps_per_epoch):
    """Warm the learning rate up linearly, then decay it on a cosine. The schedule steps once per batch."""
    warmup_steps = config["warmup_epochs"] * steps_per_epoch
    total_steps = config["epochs"] * steps_per_epoch
    warmup = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=config["warmup_start_factor"], end_factor=1.0, total_iters=warmup_steps)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, total_steps - warmup_steps), eta_min=config["min_lr"])
    return torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], milestones=[warmup_steps])


def get_alpha_values(model):
    """Read the current alpha of every DyT layer. A LayerNorm model returns an empty dict."""
    return {name: m.alpha.item() for name, m in model.named_modules() if isinstance(m, DynamicTanh)}


def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device, use_amp):
    """Train for one epoch. Returns the average loss and the top-1 accuracy, in percent."""
    model.train()
    loss_sum = torch.zeros((), device=device)
    correct = torch.zeros((), device=device)
    n_seen = 0
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        loss_sum += loss.detach() * targets.size(0)
        correct += (logits.argmax(1) == targets).sum()
        n_seen += targets.size(0)
    return (loss_sum / n_seen).item(), (correct / n_seen * 100).item()


@torch.no_grad()
def evaluate(model, loader, criterion, device, use_amp, return_outputs=False):
    """Score a loader without updating weights. Optionally also return the logits and the labels."""
    model.eval()
    loss_sum, correct, n_seen = 0.0, 0, 0
    all_logits, all_targets = [], []
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, targets)
        loss_sum += loss.item() * targets.size(0)
        correct += (logits.argmax(1) == targets).sum().item()
        n_seen += targets.size(0)
        if return_outputs:
            all_logits.append(logits.float().cpu())
            all_targets.append(targets.cpu())
    loss, acc = loss_sum / n_seen, correct / n_seen * 100
    if return_outputs:
        return loss, acc, torch.cat(all_logits), torch.cat(all_targets)
    return loss, acc


### Checkpoints and resume
After every epoch the full training state is written to `latest_checkpoint.pt`. The file is saved under a temporary name and then moved into place, so a disconnect in the middle of a save cannot leave a half-written checkpoint. `best_model.pt` keeps the weights with the highest validation accuracy. Running a training cell again after a Colab disconnect continues from the last finished epoch. Re-run the cells above it first, because the runtime itself has been wiped.


In [ ]:
def atomic_torch_save(obj, path):
    """Write a checkpoint to a temporary file, then replace the real file. A crash cannot leave a half-written checkpoint."""
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def atomic_json_dump(obj, path):
    """Write JSON the same way: a temporary file, then a replace."""
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def get_rng_state():
    """Capture the Python, NumPy, and PyTorch random states so a resumed run continues the same sequence."""
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def set_rng_state(state):
    """Put back the random states stored in a checkpoint."""
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if state["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in state["cuda"]])


def model_paths(model_key):
    """Return the checkpoint, history, and test-result paths for one model."""
    d = os.path.join(SAVE_DIR, model_key)
    return {"dir": d,
            "latest": os.path.join(d, "latest_checkpoint.pt"),
            "best": os.path.join(d, "best_model.pt"),
            "history": os.path.join(d, "history.json"),
            "test": os.path.join(d, "test_results.json")}


def run_experiment(model_key, loaders, config=CONFIG, stop_after_epochs=None):
    """Train one model, or continue it if a checkpoint is already on disk.
    The best validation weights and a full checkpoint are saved after every epoch.
    """
    spec = EXPERIMENTS[model_key]
    paths = model_paths(model_key)

    set_seed(config["seed"])
    model = build_model(spec["norm_type"], spec["alpha_init"], config).to(DEVICE)
    optimizer = build_optimizer(model, config)
    scheduler = build_scheduler(optimizer, config, steps_per_epoch=len(loaders["train"]))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    criterion = nn.CrossEntropyLoss(label_smoothing=config["label_smoothing"])

    history = {k: [] for k in ("train_loss", "train_acc", "val_loss", "val_acc", "lr",
                               "epoch_time_sec", "peak_mem_mb", "alpha")}
    start_epoch, best_val_acc = 0, 0.0

    if os.path.exists(paths["latest"]):
        ckpt = torch.load(paths["latest"], map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        scaler.load_state_dict(ckpt["scaler_state_dict"])
        history, best_val_acc = ckpt["history"], ckpt["best_val_acc"]
        start_epoch = ckpt["epoch"] + 1
        if "rng_state" in ckpt:
            set_rng_state(ckpt["rng_state"])
        print(f"[{spec['display_name']}] Resumed from epoch {ckpt['epoch'] + 1}/{config['epochs']} "
              f"(best val acc {best_val_acc:.2f}%)")

    if start_epoch >= config["epochs"]:
        print(f"[{spec['display_name']}] Training already complete. Loaded the saved history.")
        atomic_json_dump(history, paths["history"])
        return {"history": history, "best_val_acc": best_val_acc, "start_epoch": start_epoch}

    end_epoch = config["epochs"] if stop_after_epochs is None else min(config["epochs"], start_epoch + stop_after_epochs)
    print(f"[{spec['display_name']}] Training epochs {start_epoch + 1}-{end_epoch} of {config['epochs']} on {DEVICE} "
          f"| params {count_parameters(model):,}")

    for epoch in range(start_epoch, end_epoch):
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats(DEVICE)
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        train_loss, train_acc = train_one_epoch(model, loaders["train"], criterion, optimizer, scheduler,
                                                scaler, DEVICE, USE_AMP)
        val_loss, val_acc = evaluate(model, loaders["val"], criterion, DEVICE, USE_AMP)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        epoch_time = time.perf_counter() - t0
        peak_mem = torch.cuda.max_memory_allocated(DEVICE) / 1024**2 if DEVICE.type == "cuda" else 0.0

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(optimizer.param_groups[0]["lr"])
        history["epoch_time_sec"].append(epoch_time)
        history["peak_mem_mb"].append(peak_mem)
        history["alpha"].append(get_alpha_values(model))

        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc
            atomic_torch_save({"epoch": epoch, "model_state_dict": model.state_dict(), "val_acc": val_acc,
                               "model_key": model_key, "config": config}, paths["best"])

        atomic_torch_save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_val_acc": best_val_acc,
            "history": history,
            "config": config,
            "rng_state": get_rng_state(),
        }, paths["latest"])
        atomic_json_dump(history, paths["history"])

        alpha_str = ""
        if history["alpha"][-1]:
            a = list(history["alpha"][-1].values())
            alpha_str = f" | α[first/last] {a[0]:.3f}/{a[-1]:.3f}"
        print(f"  ep {epoch + 1:3d}/{config['epochs']} | train {train_loss:.4f} / {train_acc:5.2f}% | "
              f"val {val_loss:.4f} / {val_acc:5.2f}% | lr {history['lr'][-1]:.2e} | {epoch_time:5.1f}s | "
              f"{peak_mem:6.0f} MB{alpha_str}{'  *best*' if is_best else ''}")

    del model, optimizer, scheduler, scaler
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return {"history": history, "best_val_acc": best_val_acc, "start_epoch": start_epoch}


def launch_experiment(model_key):
    """Train one model to the end. In smoke-test mode, stop after epoch 1 and start again to prove resume works."""
    if DEBUG_SMOKE_TEST:
        run_experiment(model_key, loaders, stop_after_epochs=1)
        print("  -- simulated disconnect after epoch 1; re-launching --")
        result = run_experiment(model_key, loaders)
        assert result["start_epoch"] == 1, "resume did not pick up the epoch-1 checkpoint"
        assert len(result["history"]["train_loss"]) == CONFIG["epochs"]
        print(f"  Smoke test OK: resumed at epoch 2 and finished {CONFIG['epochs']} epochs.")
        return result
    return run_experiment(model_key, loaders)


### Model A — LayerNorm
Trains the baseline. About one hour on a Colab T4. If the runtime disconnects, rerun the cells above this one and then rerun this cell. It continues from the last finished epoch.


In [ ]:
results_A = launch_experiment("ln_vit")
print(f"LayerNorm-ViT best val acc: {results_A['best_val_acc']:.2f}%")

### Model B — DyT, initial α = 0.5
Same recipe as Model A. Every LayerNorm is a Dynamic Tanh layer that starts at α = 0.5. The same resume rule applies.


In [ ]:
results_B = launch_experiment("dyt_vit_a05")
print(f"DyT-ViT (α₀=0.5) best val acc: {results_B['best_val_acc']:.2f}%")

### Model C — DyT, initial α = 1.0
Same as Model B except α starts at 1.0. A larger start makes tanh flatten sooner. This is our own check of that choice. It is not a separate model from the paper. The same resume rule applies.


In [ ]:
results_C = launch_experiment("dyt_vit_a10")
print(f"DyT-ViT-Var (α₀=1.0) best val acc: {results_C['best_val_acc']:.2f}%")

## Evaluation

Each model is scored once on the 10,000 test images. The weights are the checkpoint with the best validation accuracy. The test set is not used to choose that checkpoint.


In [ ]:
def load_best_model(model_key):
    """Build the model and load the checkpoint that had the best validation accuracy."""
    spec = EXPERIMENTS[model_key]
    model = build_model(spec["norm_type"], spec["alpha_init"], CONFIG).to(DEVICE)
    ckpt = torch.load(model_paths(model_key)["best"], map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    return model.eval(), ckpt


def evaluate_on_test(model_key):
    """Score that checkpoint on the test set and save accuracy, F1, and the training cost."""
    spec, paths = EXPERIMENTS[model_key], model_paths(model_key)
    model, ckpt = load_best_model(model_key)
    criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
    test_loss, top1, logits, targets = evaluate(model, loaders["test"], criterion, DEVICE, USE_AMP, return_outputs=True)

    top5 = (logits.topk(5, dim=1).indices == targets[:, None]).any(dim=1).float().mean().item() * 100
    y_true, y_pred = targets.numpy(), logits.argmax(1).numpy()
    labels = np.arange(CONFIG["num_classes"])
    prec, rec, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    cm_fine = confusion_matrix(y_true, y_pred, labels=labels)
    cm_coarse = confusion_matrix(FINE_TO_COARSE[y_true], FINE_TO_COARSE[y_pred], labels=np.arange(20), normalize="true")

    with open(paths["history"]) as f:
        history = json.load(f)
    summary = {
        "Model": spec["display_name"],
        "Test Top-1 (%)": round(top1, 2),
        "Test Top-5 (%)": round(top5, 2),
        "Macro Precision": round(float(prec), 4),
        "Macro Recall": round(float(rec), 4),
        "Macro F1": round(float(f1_macro), 4),
        "Weighted F1": round(float(f1_weighted), 4),
        "Test Loss": round(test_loss, 4),
        "Best Val Acc (%)": round(ckpt["val_acc"], 2),
        "Best Epoch": ckpt["epoch"] + 1,
        "Total Train Time (min)": round(sum(history["epoch_time_sec"]) / 60, 2),
        "Avg Epoch Time (s)": round(float(np.mean(history["epoch_time_sec"])), 2),
        "Peak GPU Mem (MB)": round(max(history["peak_mem_mb"]), 1),
        "Params": count_parameters(model),
        "Epochs Trained": len(history["train_loss"]),
        "Test Images": len(y_true),
    }
    atomic_json_dump(summary, paths["test"])
    del model
    return summary, cm_fine, cm_coarse, history


test_summaries, cms_fine, cms_coarse, histories = {}, {}, {}, {}
for key in EXPERIMENTS:
    test_summaries[key], cms_fine[key], cms_coarse[key], histories[key] = evaluate_on_test(key)

atomic_json_dump(test_summaries, os.path.join(SAVE_DIR, "test_results.json"))
results_df = pd.DataFrame(list(test_summaries.values())).set_index("Model")
results_df.to_csv(os.path.join(SAVE_DIR, "comparison_table.csv"))
pd.set_option("display.width", 200)
display(results_df.T.astype(object))


### Not the same experiment as the paper
The paper reports ImageNet accuracy for ViT-B and ViT-L. Those numbers use a different dataset, a much larger model, and a heavier training recipe, so they cannot be compared with our accuracies. The gap between DyT and LayerNorm, under one recipe, is the number that can be discussed.


In [ ]:
paper = {"ViT-B (ImageNet-1K)": (82.3, 82.5), "ViT-L (ImageNet-1K)": (83.1, 83.6)}
ln_acc = test_summaries["ln_vit"]["Test Top-1 (%)"]
rows = [{"Setting": k, "LayerNorm Top-1": ln, "DyT Top-1": d, "Δ (DyT − LN)": round(d - ln, 2), "Source": "Reference (ImageNet-1K)"}
        for k, (ln, d) in paper.items()]
for key in ("dyt_vit_a05", "dyt_vit_a10"):
    d = test_summaries[key]["Test Top-1 (%)"]
    rows.append({"Setting": f"Compact ViT, CIFAR-100, {EXPERIMENTS[key]['display_name']}", "LayerNorm Top-1": ln_acc,
                 "DyT Top-1": d, "Δ (DyT − LN)": round(d - ln_acc, 2), "Source": "This notebook"})
paper_vs_ours = pd.DataFrame(rows).set_index("Setting")
paper_vs_ours.to_csv(os.path.join(SAVE_DIR, "paper_vs_ours.csv"))
display(paper_vs_ours)

### Training curves
Training accuracy is measured on augmented images with dropout on, so early in training it can sit below the validation accuracy.


In [ ]:
COLORS = {"ln_vit": "tab:blue", "dyt_vit_a05": "tab:orange", "dyt_vit_a10": "tab:green"}
PLOT_FILES = ["cifar100_samples.png", "dyt_vs_layernorm_mapping.png", "training_curves.png",
              "alpha_trajectories.png", "coarse_confusion_matrices.png", "top10_confused_pairs.png"]


def save_fig(fig, name):
    """Save a figure as both PNG and PDF in the plots folder."""
    fig.savefig(os.path.join(PLOT_DIR, name), dpi=200, bbox_inches="tight")
    fig.savefig(os.path.join(PLOT_DIR, name.replace(".png", ".pdf")), bbox_inches="tight")


fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for key, h in histories.items():
    ep = np.arange(1, len(h["train_loss"]) + 1)
    label = EXPERIMENTS[key]["display_name"]
    axes[0].plot(ep, h["train_loss"], color=COLORS[key], ls="--", alpha=0.8, label=f"{label} train")
    axes[0].plot(ep, h["val_loss"], color=COLORS[key], label=f"{label} val")
    axes[1].plot(ep, h["train_acc"], color=COLORS[key], ls="--", alpha=0.8, label=f"{label} train")
    axes[1].plot(ep, h["val_acc"], color=COLORS[key], label=f"{label} val")
    axes[2].plot(ep, h["lr"], color=COLORS[key], label=label)
axes[0].set(title="Cross-entropy loss (label smoothing 0.1)", xlabel="epoch", ylabel="loss")
axes[1].set(title="Top-1 accuracy", xlabel="epoch", ylabel="accuracy (%)")
axes[2].set(title="Learning rate (warmup + cosine)", xlabel="epoch", ylabel="lr")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
plt.tight_layout()
save_fig(fig, "training_curves.png")
plt.show()


### Learned α
α is free to move during training. These plots show where it ends up in the early, middle, and late blocks, and in the final norm.


In [ ]:
dyt_keys = [k for k in EXPERIMENTS if EXPERIMENTS[k]["norm_type"] == "dyt"]
depth = CONFIG["depth"]
panels = [("Early (block 0)", 0), (f"Middle (block {depth // 2})", depth // 2), (f"Late (block {depth - 1})", depth - 1)]

fig, axes = plt.subplots(1, 5, figsize=(24, 4.5))
for key in dyt_keys:
    alpha_hist = histories[key]["alpha"]
    ep = np.arange(1, len(alpha_hist) + 1)
    label = EXPERIMENTS[key]["display_name"]
    for ax, (title, b) in zip(axes[:3], panels):
        ax.plot(ep, [a[f"blocks.{b}.norm1"] for a in alpha_hist], color=COLORS[key], label=f"{label} norm1")
        ax.plot(ep, [a[f"blocks.{b}.norm2"] for a in alpha_hist], color=COLORS[key], ls="--", label=f"{label} norm2")
        ax.set_title(title)
    axes[3].plot(ep, [a["norm"] for a in alpha_hist], color=COLORS[key], label=label)
    final = alpha_hist[-1]
    axes[4].plot(range(len(final)), list(final.values()), "o-", color=COLORS[key], label=label)
    axes[4].set_xticks(range(len(final)))
    axes[4].set_xticklabels([n.replace("blocks.", "b").replace(".norm", ".n") for n in final], rotation=60, fontsize=7)
axes[3].set_title("Final norm (before head)")
axes[4].set_title("Final α per layer")
for ax in axes[:4]:
    ax.set(xlabel="epoch", ylabel="α")
    for a0 in (CONFIG["alpha_init_standard"], CONFIG["alpha_init_variation"]):
        ax.axhline(a0, color="gray", lw=0.6, ls=":")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
plt.suptitle("Learned DyT α: α₀ = 0.5 vs α₀ = 1.0", fontsize=13)
plt.tight_layout()
save_fig(fig, "alpha_trajectories.png")
plt.show()

alpha_table = pd.DataFrame({EXPERIMENTS[k]["display_name"]: histories[k]["alpha"][-1] for k in dyt_keys}).round(4)
display(alpha_table.T)

### Confusion
A 100×100 matrix is hard to read, so predictions are also grouped into the 20 superclasses. The bars under that are the 10 fine-class pairs each model mixes up most often.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(28, 9), constrained_layout=True)
for ax, key in zip(axes, EXPERIMENTS):
    im = ax.imshow(cms_coarse[key], cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"{EXPERIMENTS[key]['display_name']}  (coarse acc {np.trace(cms_coarse[key]) / 20 * 100:.1f}%)")
    ax.set_xticks(range(20)); ax.set_yticks(range(20))
    ax.set_xticklabels(COARSE_NAMES, rotation=90, fontsize=7)
    ax.set_yticklabels(COARSE_NAMES, fontsize=7)
    ax.set_xlabel("predicted superclass"); ax.set_ylabel("true superclass")
fig.colorbar(im, ax=axes, fraction=0.015, pad=0.01, label="row-normalized rate")
save_fig(fig, "coarse_confusion_matrices.png")
plt.show()


def top_confused_pairs(cm, names, k=10):
    """Return the k class pairs that the model confuses most often, ignoring correct predictions."""
    off = cm.copy()
    np.fill_diagonal(off, 0)
    flat = np.argsort(off.ravel())[::-1][:k]
    rows = []
    for idx in flat:
        i, j = divmod(int(idx), cm.shape[1])
        rows.append({"true": names[i], "predicted": names[j], "count": int(off[i, j]),
                     "% of true class": round(100 * off[i, j] / max(cm[i].sum(), 1), 1)})
    return pd.DataFrame(rows)


fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, key in zip(axes, EXPERIMENTS):
    df = top_confused_pairs(cms_fine[key], FINE_NAMES)
    df.to_csv(os.path.join(SAVE_DIR, key, "top10_confused_pairs.csv"), index=False)
    print(f"\n{EXPERIMENTS[key]['display_name']} — top-10 confused fine-class pairs")
    display(df)
    ax.barh([f"{t} → {p}" for t, p in zip(df["true"], df["predicted"])][::-1], df["count"][::-1], color=COLORS[key])
    ax.set_title(EXPERIMENTS[key]["display_name"])
    ax.set_xlabel("# test images")
    ax.tick_params(axis="y", labelsize=8)
plt.suptitle("Top-10 most confused fine-class pairs (true → predicted)", fontsize=13)
plt.tight_layout()
save_fig(fig, "top10_confused_pairs.png")
plt.show()


## Saved files
This checks that the checkpoints, tables, and plots are all in the save folder. On Colab that folder is on Google Drive, so the files are still there after a disconnect.


In [ ]:
required = [os.path.join(SAVE_DIR, f) for f in ("test_results.json", "comparison_table.csv", "paper_vs_ours.csv", "split_indices.json")]
for key in EXPERIMENTS:
    p = model_paths(key)
    required += [p["latest"], p["best"], p["history"], p["test"]]
required += [os.path.join(PLOT_DIR, f) for f in PLOT_FILES]

missing = []
print(f"Artifact checklist ({os.path.abspath(SAVE_DIR)}):")
for path in required:
    ok = os.path.exists(path)
    size = f"{os.path.getsize(path) / 1024:9.1f} KB" if ok else ""
    print(f"  [{'OK' if ok else 'MISSING':7s}] {os.path.relpath(path, SAVE_DIR):45s} {size}")
    if not ok:
        missing.append(path)

print("\nDirectory tree:")
for root, dirs, files in os.walk(SAVE_DIR):
    dirs[:] = sorted(d for d in dirs if d != "smoke_test" or DEBUG_SMOKE_TEST)
    level = 0 if root == SAVE_DIR else os.path.relpath(root, SAVE_DIR).count(os.sep) + 1
    print("  " * level + f"{os.path.basename(root) or root}/")
    for f in sorted(files):
        print("  " * (level + 1) + f)

assert not missing, f"{len(missing)} artifact(s) missing"
assert not any(f.endswith(".tmp") for _, _, fs in os.walk(SAVE_DIR) for f in fs), "stale .tmp file found"
print(f"\nAll {len(required)} artifacts present. Mode: {'SMOKE TEST' if DEBUG_SMOKE_TEST else 'FULL RUN'}.")